# EfficientNet B4 — Regression (Moen et al. 2023 reproduction)

Reproducing the B4 model from *"Age interpretation of cod otoliths using deep learning"* (Ecological Informatics 78, 2023).

**Paper methodology:**
- **Regression** with single linear output (not classification)
- **MSE** loss; accuracy = % predictions rounded to nearest integer matching label
- **Stratified 10-fold CV** on 90% of data; 10% held-out test set
- Per-fold **z-score normalization** of age labels (train mean/std); inverse transform at inference
- **Augmentation**: random rotation 0–360°, vertical flip
- **Image size**: 380×380 (B4 native)
- **Early stopping** on validation MSE
- **Target B4 results**: accuracy ~71.7%, MSE ~0.284

In [1]:
import os
os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from sklearn.model_selection import StratifiedKFold
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from collections import Counter

# ---- Device ----
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print('Using MPS (Apple Silicon GPU)')
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print('Using CUDA')
else:
    device = torch.device('cpu')
    print('Using CPU')

Using MPS (Apple Silicon GPU)


In [2]:
# ---- Config ----
DATA_DIR    = '../../otolith_images/segmented_images'
IMG_SIZE    = 380
BATCH_SIZE  = 16          # paper used 16 on A6000; fits well on MPS
N_FOLDS     = 10
TEST_FRAC   = 0.10        # 10% held-out test set
SEED        = 77
PATIENCE    = 15          # early stopping patience
MAX_EPOCHS  = 100

torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
# ---- Remove PNGs (keep only JPGs) ----
data_dir = Path(DATA_DIR)
for png in data_dir.rglob('*.png'):
    png.unlink()
    print(f'Deleted {png}')

In [4]:
# ---- Transforms (paper: rotation 0-360, vertical flip, resize 380x380) ----
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(360),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

In [5]:
# ---- Load dataset and extract integer age labels ----
full_dataset = datasets.ImageFolder(root=DATA_DIR)
class_names  = full_dataset.classes
print(f'Folder classes: {class_names}')

# Map folder index -> integer age (folder names are age strings)
all_ages = np.array([int(class_names[label]) for _, label in full_dataset])
print(f'Age range: {all_ages.min()}–{all_ages.max()}, N={len(all_ages)}')
print(f'Age distribution: {dict(sorted(Counter(all_ages).items()))}')

# ---- Stratified train/test split (90/10) ----
from sklearn.model_selection import train_test_split

indices = np.arange(len(full_dataset))
train_idx, test_idx = train_test_split(
    indices, test_size=TEST_FRAC, random_state=SEED, stratify=all_ages
)
print(f'\nTrain set: {len(train_idx)} | Test set: {len(test_idx)}')

Folder classes: ['1', '10', '11', '12', '13', '14', '15', '16', '17', '2', '3', '4', '5', '6', '7', '8', '9']
Age range: 1–17, N=8619
Age distribution: {np.int64(1): 106, np.int64(2): 223, np.int64(3): 345, np.int64(4): 763, np.int64(5): 1691, np.int64(6): 1940, np.int64(7): 1477, np.int64(8): 926, np.int64(9): 736, np.int64(10): 220, np.int64(11): 79, np.int64(12): 50, np.int64(13): 26, np.int64(14): 17, np.int64(15): 11, np.int64(16): 7, np.int64(17): 2}

Train set: 7757 | Test set: 862


In [ ]:
# ---- Dataset wrapper for regression with z-score normalization ----
class OtolithRegressionDataset(Dataset):
    """Wraps ImageFolder subset, returns (image, normalized_age_float)."""
    def __init__(self, base_dataset, indices, age_array, transform, age_mean=None, age_std=None):
        self.base      = base_dataset
        self.indices   = indices
        self.ages      = age_array[indices].astype(np.float32)
        self.transform = transform
        # Compute or use provided normalization stats
        self.age_mean = age_mean if age_mean is not None else self.ages.mean()
        self.age_std  = age_std  if age_std  is not None else self.ages.std()

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, _ = self.base[self.indices[idx]]
        img = self.transform(img)
        age_norm = (self.ages[idx] - self.age_mean) / self.age_std
        return img, torch.tensor(age_norm, dtype=torch.float32)

    def denormalize(self, pred):
        """Inverse z-score transform."""
        return pred * self.age_std + self.age_mean

In [ ]:
# ---- Model factory ----
def make_model():
    """EfficientNet B4 with single linear regression head."""
    model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(in_features, 1),  # single regression output
    )
    return model.to(device)

# Quick check
_m = make_model()
print(f'Parameters: {sum(p.numel() for p in _m.parameters()):,}')
del _m

In [ ]:
# ---- Training helpers ----
criterion = nn.MSELoss()


def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, total = 0.0, 0
    for imgs, ages in tqdm(loader, desc='Train', leave=False):
        imgs = imgs.to(device, non_blocking=True)
        ages = ages.to(device, non_blocking=True)
        optimizer.zero_grad()
        preds = model(imgs).squeeze(1)
        loss = criterion(preds, ages)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        total += imgs.size(0)
    if device.type == 'mps':
        torch.mps.synchronize()
    return total_loss / total


@torch.no_grad()
def val_epoch(model, loader):
    model.eval()
    total_loss, total = 0.0, 0
    for imgs, ages in tqdm(loader, desc='Val', leave=False):
        imgs = imgs.to(device, non_blocking=True)
        ages = ages.to(device, non_blocking=True)
        preds = model(imgs).squeeze(1)
        loss = criterion(preds, ages)
        total_loss += loss.item() * imgs.size(0)
        total += imgs.size(0)
    if device.type == 'mps':
        torch.mps.synchronize()
    return total_loss / total


@torch.no_grad()
def predict(model, loader):
    """Return arrays of (predictions_normalized, targets_normalized)."""
    model.eval()
    all_preds, all_targets = [], []
    for imgs, ages in loader:
        imgs = imgs.to(device, non_blocking=True)
        preds = model(imgs).squeeze(1)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(ages.numpy())
    return np.concatenate(all_preds), np.concatenate(all_targets)


def compute_metrics(preds_raw, targets_raw):
    """Compute MSE and accuracy (rounded prediction == integer label)."""
    mse = float(np.mean((preds_raw - targets_raw) ** 2))
    acc = float(np.mean(np.round(preds_raw) == np.round(targets_raw)))
    return mse, acc

In [ ]:
# ---- 10-Fold Cross-Validation on training set ----
train_ages = all_ages[train_idx]
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_results = []  # stores (best_val_mse, best_state_dict, age_mean, age_std)

for fold, (fold_train_rel, fold_val_rel) in enumerate(skf.split(train_idx, train_ages)):
    print(f'\n{"="*60}')
    print(f'FOLD {fold+1}/{N_FOLDS}')
    print(f'{"="*60}')

    fold_train_idx = train_idx[fold_train_rel]
    fold_val_idx   = train_idx[fold_val_rel]

    # Per-fold z-score normalization (computed on fold train only)
    ds_train = OtolithRegressionDataset(
        full_dataset, fold_train_idx, all_ages, train_transform
    )
    ds_val = OtolithRegressionDataset(
        full_dataset, fold_val_idx, all_ages, val_transform,
        age_mean=ds_train.age_mean, age_std=ds_train.age_std
    )
    print(f'  Train: {len(ds_train)}, Val: {len(ds_val)}')
    print(f'  Age norm: mean={ds_train.age_mean:.2f}, std={ds_train.age_std:.2f}')

    train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, pin_memory=True)
    val_loader   = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=True)

    # Fresh model each fold
    model = make_model()
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    best_val_mse = float('inf')
    best_state   = None
    wait = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        tr_loss = train_epoch(model, train_loader, optimizer)
        vl_loss = val_epoch(model, val_loader)
        scheduler.step()

        marker = ''
        if vl_loss < best_val_mse:
            best_val_mse = vl_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
            marker = ' *'
        else:
            wait += 1

        if epoch % 5 == 0 or marker:
            print(f'  Epoch {epoch:3d}  train_mse={tr_loss:.4f}  val_mse={vl_loss:.4f}{marker}')

        if wait >= PATIENCE:
            print(f'  Early stopping at epoch {epoch}')
            break

        if device.type == 'mps':
            torch.mps.empty_cache()

    fold_results.append({
        'best_val_mse': best_val_mse,
        'state_dict': best_state,
        'age_mean': ds_train.age_mean,
        'age_std': ds_train.age_std,
    })
    print(f'  Best val MSE (normalized): {best_val_mse:.4f}')

    # Cleanup
    del model, optimizer, scheduler
    if device.type == 'mps':
        torch.mps.empty_cache()

print(f'\n{"="*60}')
print(f'CV done. Mean best val MSE (norm): {np.mean([r["best_val_mse"] for r in fold_results]):.4f}')

In [ ]:
# ---- Evaluate best fold model on held-out test set ----
# Paper: select the fold with best val MSE, evaluate on test set

best_fold_idx = int(np.argmin([r['best_val_mse'] for r in fold_results]))
best = fold_results[best_fold_idx]
print(f'Best fold: {best_fold_idx+1} (val MSE={best["best_val_mse"]:.4f})')

# Load best model
model = make_model()
model.load_state_dict(best['state_dict'])
model.eval()

# Test dataset with same normalization as best fold
ds_test = OtolithRegressionDataset(
    full_dataset, test_idx, all_ages, val_transform,
    age_mean=best['age_mean'], age_std=best['age_std']
)
test_loader = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=True)

# Predict and denormalize
preds_norm, targets_norm = predict(model, test_loader)
preds_raw   = ds_test.denormalize(preds_norm)
targets_raw = ds_test.denormalize(targets_norm)

mse, acc = compute_metrics(preds_raw, targets_raw)
print(f'\nTest MSE:      {mse:.3f}')
print(f'Test Accuracy: {acc*100:.1f}%')
print(f'\n(Paper B4-min target: MSE≈0.284, Acc≈71.7%)')

In [ ]:
# ---- Per-age-class accuracy (violin plot like paper Fig. 4) ----
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Scatter: predicted vs labelled
ax = axes[0]
ax.scatter(targets_raw, preds_raw, alpha=0.3, s=10)
lims = [all_ages.min()-0.5, all_ages.max()+0.5]
ax.plot(lims, lims, 'r--', lw=1)
ax.set_xlabel('Labelled age')
ax.set_ylabel('Predicted age')
ax.set_title(f'Test set: MSE={mse:.3f}, Acc={acc*100:.1f}%')

# Per-age accuracy bar chart
ax = axes[1]
unique_ages = sorted(np.unique(np.round(targets_raw).astype(int)))
age_accs = []
for age in unique_ages:
    mask = np.round(targets_raw).astype(int) == age
    if mask.sum() > 0:
        age_acc = np.mean(np.round(preds_raw[mask]) == age)
        age_accs.append(age_acc * 100)
    else:
        age_accs.append(0)
bars = ax.bar(unique_ages, age_accs, color='steelblue')
ax.set_xlabel('Age')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Per-age accuracy on test set')
for bar, acc_val, age in zip(bars, age_accs, unique_ages):
    n = int(np.sum(np.round(targets_raw).astype(int) == age))
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{acc_val:.0f}%\nn={n}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# ---- Save best model ----
save_path = 'efficientnetb4_regression_otolith.pth'
torch.save({
    'state_dict': best['state_dict'],
    'age_mean': best['age_mean'],
    'age_std': best['age_std'],
    'test_mse': mse,
    'test_acc': acc,
    'best_fold': best_fold_idx,
}, save_path)
print(f'Saved to {save_path}')